# Isolement des communes françaises : des distances aux conclusions

**Question.** Quelles communes de France métropolitaine sont les plus éloignées des services du quotidien ?

**Ce notebook** va des données brutes jusqu'aux conclusions : indice d'isolement, cartes, liens avec la densité et le revenu, typologie des communes et autocorrélation spatiale.

### Comment lire et utiliser ce notebook

- Il se lance depuis la **racine du projet** (le dossier qui contient `src/` et `data/`).
- Chaque étape commence par une cellule de texte en cinq rubriques : **Objectif**, **Entrées**, **Ce que fait le code**, **Sorties**, **À vérifier**.
- Les cellules marquées **« À ADAPTER »** contiennent des noms de fichiers ou de colonnes que seule l'inspection de vos fichiers permet de remplir.
- La logique de calcul vit dans `src/` (fichiers `.py` testés). Le notebook les appelle et raconte l'analyse.
- Pour rejouer tout le rapport : *Kernel → Restart & Run All*.

### Plan

| # | Étape | Produit |
|---|---|---|
| 0 | Mise en place | imports, chemins, paramètres |
| 1 | Données | fichiers INSEE téléchargés, fichiers manuels repérés |
| 2 | Nettoyage de la BPE | `data/interim/equipements.parquet` |
| 3 | Table des communes | une ligne par commune ou arrondissement : centre, surface, population, densité |
| 4 | Présence des services | tableau commune × service (vrai / faux) |
| 5 | Distances | tableau commune × service, en km |
| 6 | Table finale | `data/processed/communes_distances.parquet` |
| 7 | Indice d'isolement | quatre jeux de poids, trois variantes de robustesse |
| 8 | Statistiques descriptives | tableaux et graphiques |
| 9 | Cartes | indice et six services |
| 10 | Densité | corrélation et classes de densité |
| 11 | Revenu | fusion Filosofi, corrélation, test de groupes |
| 12 | Typologie | profils d'isolement (ACP et k-means) |
| 13 | Autocorrélation spatiale | I de Moran et cartes LISA (bonus) |
| 14 | Conclusion | tableau final, limites, fichiers produits |

## 0. Mise en place

**Objectif.** Regrouper au même endroit les imports, les chemins et les paramètres de l'analyse, pour qu'un changement se fasse à une seule place.

**Entrées.** Aucune.

**Ce que fait le code.**
1. Importe les bibliothèques (`pandas` pour les tableaux, `numpy` pour le calcul, `matplotlib` pour les graphiques).
2. Vérifie que Jupyter a été lancé depuis la racine du projet, en cherchant le dossier `src/`.
3. Ajoute `src/` aux dossiers où Python cherche les modules, puis importe nos modules : `download`, `prepare_bpe`, `communes`, `distances`, `indice`, `descriptif`.
4. Définit les chemins `data/raw`, `data/interim`, `data/processed` et le dossier `figures/`.
5. Fixe les paramètres de l'analyse : seuils de distance pour la population éloignée, qualités de position jugées fiables, services à cartographier.

**Sorties.** Des variables utilisées dans tout le notebook. Le dossier `figures/` est créé s'il n'existe pas.

**À vérifier.** La dernière ligne affiche la racine du projet. Si une erreur dit que `src/` est introuvable, relancez Jupyter depuis le bon dossier.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. On doit être à la racine du projet (là où se trouve le dossier src/)
RACINE = Path.cwd()
if not (RACINE / "src").exists():
    raise FileNotFoundError("Lancez Jupyter depuis la racine du projet (le dossier qui contient src/).")
sys.path.insert(0, str(RACINE / "src"))

# 2. Nos modules (voir le dossier src/)
import communes as cm
import descriptif as dsc
import distances as dist
import download
import indice as idx
import prepare_bpe

# 3. Chemins
RAW = RACINE / "data" / "raw"
INTERIM = RACINE / "data" / "interim"
PROCESSED = RACINE / "data" / "processed"
FIGURES = RACINE / "figures"
for dossier in (RAW, INTERIM, PROCESSED, FIGURES):
    dossier.mkdir(parents=True, exist_ok=True)

# 4. Paramètres de l'analyse
SEUILS_KM = [5, 10, 20]                      # « population à plus de X km d'un service »
QUALITES_FIABLES = ["A", "B"]                # positions jugées fiables (variante de robustesse)
SERVICES_CARTES = ["pharmacie", "medecin", "boulangerie", "ecole", "urgences", "gare"]
INDICE_PRINCIPAL = "egaux"                   # l'indice présenté dans les cartes (poids égaux)

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 60)
print("Racine du projet :", RACINE)

## 1. Données

### 1.1 Téléchargement automatique des fichiers INSEE

**Objectif.** Récupérer par du code les 5 fichiers INSEE (la BPE 2025 et sa documentation), pour que n'importe qui puisse refaire le projet depuis zéro.

**Entrées.** Aucune (les adresses sont écrites dans `src/download.py`).

**Ce que fait le code.** Appelle `download.main()`. Pour chaque fichier : saute s'il est déjà présent, sinon le télécharge par blocs vers un fichier temporaire, puis vérifie qu'il s'agit bien d'un parquet ou d'un CSV et non d'une page d'erreur.

**Sorties.** `BPE25.parquet`, `TYPEQU_2025.csv`, `BPE25_dessin_fichier.csv`, `BPE25_varmod.csv`, `BPE25_table_passage.csv` dans `data/raw/`.

**À vérifier.** Le message final doit être « Téléchargements automatiques terminés » et la cellule doit renvoyer `0`. Sur une deuxième exécution, tout doit afficher « déjà présent ».

In [ ]:
download.main()

### 1.2 Les fichiers à télécharger à la main

**Objectif.** Constater quels fichiers sont présents dans `data/raw/`. Trois fichiers n'ont pas d'adresse stable et se téléchargent à la main :

| Fichier | Où le trouver | À quoi il sert |
|---|---|---|
| **Contours des communes et arrondissements** (IGN Admin Express, édition COG 2025) | https://geoservices.ign.fr/adminexpress | centres des communes (points de départ des distances), surfaces, cartes |
| **Population municipale 2023** (géographie 2025) | https://www.data.gouv.fr/datasets/populations-de-reference | densité, part de la population éloignée |
| **Revenus Filosofi** (niveau commune) | https://www.insee.fr (rechercher « Filosofi ») | lien isolement et revenu (étape suivante) |

L'archive Admin Express est un fichier `.7z` : décompressez-la **dans `data/raw/`** (le dossier obtenu peut rester tel quel).

**Ce que fait le code.** Liste le contenu de `data/raw/` : pour chaque élément, sa taille (ou son nombre de fichiers s'il s'agit d'un dossier).

**À vérifier.** Après vos téléchargements, vous devez voir les 5 fichiers INSEE, un dossier Admin Express et le fichier de population. Le fichier Filosofi peut attendre l'étape suivante.

In [ ]:
for element in sorted(RAW.iterdir()):
    if element.name.startswith("."):
        continue
    if element.is_dir():
        nombre = sum(1 for p in element.rglob("*") if p.is_file())
        print(f"[dossier] {element.name}  ({nombre} fichiers)")
    else:
        print(f"[fichier] {element.name}  ({element.stat().st_size / 1e6:.1f} Mo)")

### 1.3 Les contours IGN

**Objectif.** Repérer les fichiers cartographiques d'Admin Express et choisir les deux couches dont on a besoin : **COMMUNE** et **ARRONDISSEMENT_MUNICIPAL**.

**Pourquoi les deux ?** La BPE range les équipements de Paris, Lyon et Marseille **par arrondissement** (par exemple une boulangerie du 3ᵉ arrondissement de Marseille a pour code `13203`). La couche COMMUNE contient ces trois villes d'un seul tenant (`75056`, `69123`, `13055`) : on prendra donc les arrondissements à leur place.

**Ce que fait le code.** La première cellule cherche tous les fichiers `.gpkg` (GeoPackage) et `.shp` (shapefile) sous `data/raw/` et les affiche. La seconde est **à adapter** : on y indique le chemin choisi.

**À vérifier.** Selon l'édition d'Admin Express, vous aurez soit **un seul GeoPackage** (`.gpkg`) qui contient toutes les couches, soit **un shapefile par couche**. Remplissez le cas qui correspond et laissez l'autre à `None`.

In [ ]:
for fichier in cm.trouver_fichiers(RAW):
    print(fichier.relative_to(RAW))

In [ ]:
# ----------------------- À ADAPTER d'après la liste affichée ci-dessus -----------------------
# Cas 1 : un seul GeoPackage contenant toutes les couches
CHEMIN_GPKG = None                          # exemple : RAW / "ADMIN-EXPRESS_.../ADE_3-2_FXX.gpkg"
COUCHE_COMMUNES = "commune"                 # nom de la couche (voir la liste affichée plus bas)
COUCHE_ARRONDISSEMENTS = "arrondissement_municipal"

# Cas 2 : un shapefile par couche
CHEMIN_SHP_COMMUNES = None                  # exemple : RAW / "..." / "COMMUNE.shp"
CHEMIN_SHP_ARRONDISSEMENTS = None           # exemple : RAW / "..." / "ARRONDISSEMENT_MUNICIPAL.shp"
# ---------------------------------------------------------------------------------------------

if CHEMIN_GPKG is not None:
    print("Couches disponibles dans le GeoPackage :", cm.lister_couches(CHEMIN_GPKG))
    gdf_com = cm.lire_couche(CHEMIN_GPKG, COUCHE_COMMUNES)
    gdf_arm = cm.lire_couche(CHEMIN_GPKG, COUCHE_ARRONDISSEMENTS)
elif CHEMIN_SHP_COMMUNES is not None and CHEMIN_SHP_ARRONDISSEMENTS is not None:
    gdf_com = cm.lire_couche(CHEMIN_SHP_COMMUNES)
    gdf_arm = cm.lire_couche(CHEMIN_SHP_ARRONDISSEMENTS)
else:
    raise ValueError("Renseignez CHEMIN_GPKG, ou bien les deux chemins de shapefiles.")

print(f"\nCommunes : {len(gdf_com):,} lignes ; colonnes : {gdf_com.columns.tolist()}")
print(f"Arrondissements : {len(gdf_arm):,} lignes ; colonnes : {gdf_arm.columns.tolist()}")
print("Système de coordonnées :", gdf_com.crs)

### 1.4 La population

**Objectif.** Disposer de la population de chaque commune (et arrondissement) pour calculer la **densité** (habitants par km²) et la **part de la population éloignée** des services.

**Deux possibilités.**
- **Fichier INSEE « populations de référence »** (recommandé : population municipale 2023, géographie 2025). Son format varie (CSV ou Excel, quelques lignes d'en-tête parfois) : on le regarde d'abord, puis on renseigne le nom des deux colonnes utiles.
- **Repli** : si `FICHIER_POPULATION` reste à `None`, on utilise la colonne de population contenue dans les contours IGN, quand elle existe. Son millésime est alors à vérifier et à citer dans le rapport.

**Ce que fait le code.** La fonction `lire_tableau` lit un CSV (séparateur deviné) ou un Excel en gardant tout en **texte**, pour ne pas perdre le zéro initial des codes. Si un fichier est indiqué, on affiche ses premières lignes.

**À vérifier.** Repérez la colonne du **code commune** et la colonne de la **population municipale**. Si l'en-tête commence plus bas, ajoutez `{"skiprows": N}` dans `POP_OPTIONS`.

In [ ]:
# ----------------------- À ADAPTER -----------------------
FICHIER_POPULATION = None      # exemple : RAW / "population_municipale_2023.xlsx"
POP_COL_CODE = None            # exemple : "CODGEO"
POP_COL_VALEUR = None          # exemple : "PMUN"
POP_OPTIONS = {}               # exemple : {"skiprows": 5}   ou   {"sep": ";"}
# ---------------------------------------------------------


def lire_tableau(chemin, **options):
    """Lit un CSV ou un Excel, tout en texte (les codes commune gardent leur zéro initial)."""
    if Path(chemin).suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(chemin, dtype=str, **options)
    parametres = {"sep": None, "engine": "python"}
    parametres.update(options)
    return pd.read_csv(chemin, dtype=str, **parametres)


if FICHIER_POPULATION is not None:
    population_brute = lire_tableau(FICHIER_POPULATION, **POP_OPTIONS)
    print(population_brute.columns.tolist())
    display(population_brute.head())
else:
    print("Pas de fichier de population indiqué : on utilisera celle des contours IGN (voir 3.3).")

## 2. Nettoyage de la BPE

### 2.1 Les 19 services : lesquels on mélange, et pourquoi

La BPE décrit **33 types d'équipements** (un code `TYPEQU` chacun). Pour mesurer l'isolement, on raisonne en **services** : un service correspond à un **besoin d'habitant**. Quand plusieurs types répondent au même besoin, on les **regroupe** : la commune est « équipée » dès qu'elle possède **au moins un** des types du service.

**10 services restent seuls** (un seul type) :

| Service | Type | Libellé BPE |
|---|---|---|
| boulangerie | B207 | boulangerie-pâtisserie |
| pharmacie | D307 | pharmacie |
| medecin | D265 | médecin généraliste |
| dentiste | D277 | chirurgien-dentiste |
| kine | D279 | masseur-kinésithérapeute |
| infirmier | D281 | infirmier |
| urgences | D106 | urgences |
| maternite | D107 | maternité |
| college | C201 | collège |
| france_services | A128 | France Services |

**9 services regroupent plusieurs types** :

| Service | Types mélangés | Pourquoi les mélanger |
|---|---|---|
| alimentation | B201 supérette, B202 épicerie, B105 supermarché et multi-commerce | trois formats pour faire ses courses de base |
| sante_proximite | D108 centre de santé, D113 maison de santé pluridisciplinaire | deux structures de soins de premier recours |
| ecole | C107 maternelle, C108 primaire, C109 élémentaire | même besoin : scolariser un jeune enfant |
| lycee | C301 général et technologique, C302 professionnel | même niveau d'enseignement |
| poste | A206 bureau de poste, A207 relais poste, A208 agence postale | trois formes de présence postale |
| petite_enfance | D502 accueil du jeune enfant, D509 micro-crèche | même besoin : faire garder un enfant |
| personnes_agees | D401 hébergement, D402 soins à domicile, D403 aide à domicile | **mélange discutable** : un lieu et des services qui se déplacent chez la personne |
| proximite_sociale | D506 centres sociaux, D508 espaces de vie sociale | deux lieux d'accueil social local |
| gare | E107 nationale, E108 régionale, E109 locale | même besoin : prendre le train |

**Ce qui n'est volontairement PAS mélangé.** Médecin, dentiste, kiné et infirmier restent séparés : ce sont des soins différents, qu'un habitant ne substitue pas l'un à l'autre. Les mélanger masquerait des écarts.

**Deux fragilités, testées plus loin.**
1. **« Personnes âgées »** mélange des lieux (hébergement) et des services à domicile, pour lesquels la distance a un autre sens : une variante de l'indice l'exclut (étape 7).
2. **La santé pèse 8 services sur 19**, soit 42 % d'un indice à poids égaux : un jeu de poids **équilibre les thèmes** (étape 7).

Le dictionnaire `SERVICES` dans `src/prepare_bpe.py` est le **seul endroit** où ce panier est défini : pour le modifier, on ne touche que lui.

### 2.2 Filtrer la BPE sur le panier

**Objectif.** Passer des 2 921 770 lignes et 95 colonnes de la BPE à un petit fichier propre : seulement les équipements du panier, en métropole, avec les 6 colonnes utiles.

**Entrées.** `data/raw/BPE25.parquet`.

**Ce que fait le code.** Appelle `prepare_bpe.main()` qui : (1) garde la métropole (`EPSG = 2154`, c'est-à-dire coordonnées Lambert-93) ; (2) garde les 33 types du panier ; (3) range chaque type dans son service ; (4) **conserve les équipements sans coordonnées** (ils servent à la présence) ; (5) écrit `data/interim/equipements.parquet` et affiche un tableau de contrôle par service.

**Sorties.** `data/interim/equipements.parquet` : colonnes `depcom`, `service`, `typequ`, `x`, `y`, `qualite_xy`.

**À vérifier.** Dans le tableau affiché : 19 services, aucun avec 0 équipement, et une part « avec coordonnées » élevée pour chacun. **Collez-moi ce tableau** s'il vous semble étrange.

In [ ]:
prepare_bpe.main()

### 2.3 Charger et contrôler les équipements

**Objectif.** Relire le fichier propre et vérifier qu'il respecte nos hypothèses avant d'aller plus loin.

**Ce que fait le code.**
1. Charge `equipements.parquet` dans un tableau `equipements` et dresse la liste `services` (les 19 noms, dans l'ordre du dictionnaire).
2. Contrôle que les 19 services sont bien présents.
3. Contrôle que **tous les codes commune font 5 caractères** (sinon une fusion échouera silencieusement).
4. Affiche la répartition des qualités de position (`A`, `B`, `M`, `_U`, `_Z`) et la part d'équipements sans coordonnées.

**À vérifier.** Les deux contrôles (`assert`) ne doivent rien afficher. Les qualités doivent ressembler à ce qu'on a vu en inspectant la BPE : `B` très majoritaire, environ 4 % de lignes sans coordonnées.

In [ ]:
equipements = pd.read_parquet(INTERIM / "equipements.parquet")
services = list(prepare_bpe.SERVICES)

assert set(equipements["service"]) == set(services), "Un service du panier n'a aucun équipement."
assert (equipements["depcom"].str.len() == 5).all(), "Des codes commune ne font pas 5 caractères."

print(f"{len(equipements):,} équipements, {len(services)} services")
print("\nQualité de la position :")
print(equipements["qualite_xy"].value_counts(dropna=False))
print(f"\nSans coordonnées : {equipements['x'].isna().mean():.1%}")

## 3. Table des communes

### 3.1 Extraire un centre, une surface et une population de chaque contour

**Objectif.** Donner à chaque commune (et arrondissement) : un **point de départ** pour les distances, une **surface** et une **population**.

**Entrées.** Les deux couches `gdf_com` et `gdf_arm` lues à l'étape 1.3.

**Ce que fait le code.**
1. Repère les colonnes utiles de chaque couche par leur nom (code, nom, population) : les noms exacts varient selon l'édition, la fonction `choisir_colonne` essaie plusieurs noms possibles et les affiche.
2. Transforme chaque couche en simple tableau avec `geometrie_vers_table` : `code`, `nom`, `population`, `x` et `y` (le **centroïde** du polygone, en mètres Lambert-93), `surface_km2` (aire du polygone).
3. Pour les arrondissements, retient aussi la colonne **parent** : le code de la ville entière (`75056`, `13055` ou `69123`).

**Sorties.** Deux tableaux `tab_com` et `tab_arm`.

**À vérifier.** Les noms de colonnes choisis s'affichent. Si une colonne **obligatoire** manque, une erreur donne la liste des colonnes disponibles : **collez-la moi**.

In [ ]:
# Colonnes utiles (les noms varient selon l'édition d'Admin Express)
col_code_com = cm.choisir_colonne(gdf_com.columns, ["insee_com", "code_insee", "codgeo"])
col_nom_com = cm.choisir_colonne(gdf_com.columns, ["nom", "nom_com", "libgeo"], obligatoire=False)
col_pop_com = cm.choisir_colonne(gdf_com.columns, ["population", "pmun", "pop"], obligatoire=False)

col_code_arm = cm.choisir_colonne(gdf_arm.columns, ["insee_arm", "code_insee", "codgeo"])
col_parent_arm = cm.choisir_colonne(gdf_arm.columns, ["insee_com", "com", "code_commune"])
col_nom_arm = cm.choisir_colonne(gdf_arm.columns, ["nom", "nom_arm", "libgeo"], obligatoire=False)
col_pop_arm = cm.choisir_colonne(gdf_arm.columns, ["population", "pmun", "pop"], obligatoire=False)

print("Communes        : code =", col_code_com, "| nom =", col_nom_com, "| population =", col_pop_com)
print("Arrondissements : code =", col_code_arm, "| ville mère =", col_parent_arm,
      "| nom =", col_nom_arm, "| population =", col_pop_arm)

tab_com = cm.geometrie_vers_table(gdf_com, col_code_com, col_nom_com, col_pop_com)
tab_arm = cm.geometrie_vers_table(gdf_arm, col_code_arm, col_nom_arm, col_pop_arm, col_parent_arm)
display(tab_com.head(3))
display(tab_arm.head(3))

### 3.2 Assembler la base : communes + arrondissements

**Objectif.** Obtenir **une seule table de base**, où Paris, Lyon et Marseille apparaissent par arrondissement (comme dans la BPE) et jamais en double.

**Ce que fait le code.** `cm.assembler_base` :
1. retire de la table des communes les **villes mères** (les valeurs de la colonne `parent` des arrondissements) ;
2. ajoute les arrondissements ;
3. retire les départements d'outre-mer (codes commençant par `97` ou `98`) ;
4. vérifie que tous les codes font 5 caractères et qu'**aucun code n'est en double** ;
5. calcule la **densité** = population ÷ surface.

**Sorties.** `base` : une ligne par commune ou arrondissement de métropole, triée par code.

**À vérifier.**
- Les villes remplacées doivent être exactement `{75056, 13055, 69123}` (contrôle automatique).
- Le nombre de lignes est de l'ordre de **35 000** (communes de métropole, hors trois villes, plus 45 arrondissements).
- Le nombre de populations manquantes s'affiche : il doit être nul ou très faible.

In [ ]:
base = cm.assembler_base(tab_com, tab_arm)
villes_meres = set(tab_arm["parent"])

print("Villes remplacées par leurs arrondissements :", sorted(villes_meres))
assert villes_meres == {"75056", "13055", "69123"}, "Les villes mères ne sont pas Paris, Lyon et Marseille."

print(f"{len(base):,} communes et arrondissements en métropole")
print("Population manquante :", int(base["population"].isna().sum()))
display(base.describe().round(2))

### 3.3 Population : fichier INSEE ou repli sur les contours

**Objectif.** Remplacer, si on dispose du fichier INSEE, la population des contours par la **population municipale 2023**.

**Ce que fait le code.** Si `FICHIER_POPULATION` est renseigné, `cm.fusionner_population` fait une **jointure à gauche** sur le code commune : aucune commune de la base ne disparaît, les codes du fichier sont remis sur 5 caractères en texte, et la densité est recalculée. Un **journal** compte : les communes de la base sans population dans le fichier, et les codes du fichier absents de la base. Sinon, on garde la population des contours.

**À vérifier.** Dans le journal, `communes_sans_population_ext` et `codes_ext_hors_base` doivent être **proches de zéro**. Un grand nombre signale un décalage de millésime géographique ou un problème de format des codes.

In [ ]:
if FICHIER_POPULATION is not None:
    base, journal = cm.fusionner_population(base, population_brute, POP_COL_CODE, POP_COL_VALEUR)
    print("Journal de la fusion de la population :", journal)
else:
    print("Population des contours IGN conservée.")

print("Population manquante après fusion :", int(base["population"].isna().sum()))
print(f"Population totale de la base : {base['population'].sum():,.0f} habitants")

### 3.4 Polygones pour les cartes, et lien avec la BPE

**Objectif.** (1) Préparer les polygones qui serviront aux cartes ; (2) vérifier que les codes de la BPE et ceux de la base se correspondent.

**Ce que fait le code.**
1. `cm.geometries_base` construit les polygones de la base (communes sans les trois villes mères, plus les arrondissements) et on contrôle qu'ils portent exactement les mêmes codes que `base`.
2. `dist.controle_codes` compte les équipements dont le code commune est **inconnu** de la base. On compte aussi les communes de la base **sans aucun équipement du panier**.

**À vérifier.**
- `equipements_sans_commune` doit être **0 ou presque**. Sinon, un millésime géographique diffère entre la BPE et les contours : **collez-moi le résultat**.
- Les communes sans aucun équipement du panier sont normales (petits villages) : elles auront simplement de grandes distances partout.

In [ ]:
geo = cm.geometries_base(gdf_com, gdf_arm, col_code_com, col_code_arm, villes_meres)
assert set(geo["code"]) == set(base["code"]), "Les polygones et la table de base n'ont pas les mêmes codes."

print("Codes BPE absents de la base :", dist.controle_codes(equipements, base["code"]))
sans_equipement = ~base["code"].isin(equipements["depcom"])
print(f"Communes sans aucun équipement du panier : {int(sans_equipement.sum()):,} sur {len(base):,}")

## 4. Présence des services

**Objectif.** Savoir, pour chaque commune et chaque service, si **au moins un équipement** du service se trouve dans la commune. C'est la première moitié de la règle de distance : *service présent → distance 0*.

**Pourquoi la présence se décide avec le code commune.** Chaque ligne de la BPE porte le code de sa commune (`depcom`), même quand ses coordonnées manquent. Les 4 % d'équipements sans coordonnées comptent donc ici, alors qu'ils seraient inutilisables pour une distance.

**Entrées.** `equipements` (colonnes `depcom` et `service`) et la liste des codes de `base`.

**Ce que fait le code.** `dist.presence_par_service` crée un tableau de booléens (une ligne par commune, une colonne par service), initialement tout à `False`, puis met `True` aux couples (commune, service) qui existent dans la BPE.

**Sorties.** `presence` : tableau commune × service.

**À vérifier.** On affiche la **part de communes équipées par service**. L'ordre de grandeur attendu : élevé pour la boulangerie, l'école ou le médecin, faible pour les urgences, la maternité, les gares et le lycée. Si un service affiche 0 % ou 100 %, il y a un problème.

In [ ]:
presence = dist.presence_par_service(base["code"], equipements, services)
part_equipees = (presence.mean() * 100).round(1).sort_values()
display(part_equipees.to_frame("communes équipées (%)"))

## 5. Distances au service le plus proche

**Objectif.** Calculer, pour chaque commune et chaque service, **un seul nombre** : la distance (en km) à laquelle il faut aller pour trouver le service.

**La règle.**
1. **Service présent dans la commune → distance 0.**
2. **Sinon** → distance à vol d'oiseau entre le **centre de la commune** et l'équipement **le plus proche** qui a des coordonnées. On ne garde que la plus petite des distances : on n'additionne ni ne moyenne les équipements.

**Entrées.** `base` (centres `x`, `y`), `equipements` (positions), `services`.

**Ce que fait le code.** Pour chacun des 19 services, `dist.distances_par_service` :
1. rassemble les équipements du service qui ont des coordonnées ;
2. construit un **arbre k-d** (`cKDTree` de SciPy) avec leurs positions : cette structure range les points dans l'espace pour trouver le plus proche sans tout comparer (35 000 communes × 550 000 équipements représenteraient près de 20 milliards de comparaisons) ;
3. interroge l'arbre avec le centre de chaque commune : on obtient la distance en mètres au plus proche, convertie en **kilomètres** ;
4. remplace par **0** les distances des communes où le service est présent.

**Sorties.** `distances` : tableau commune × service en km (index = code commune), et `presence`.

**À vérifier.**
- Aucune distance manquante (on affiche le décompte par service, qui doit être vide).
- Toute distance d'une commune équipée vaut bien 0 (contrôle automatique).
- La table de résumé : la médiane est 0 pour les services très répandus ; le maximum reste de l'ordre de quelques dizaines de km pour les services courants. Un maximum de plusieurs centaines de km pour un service courant signale une erreur de coordonnées.

In [ ]:
distances, presence = dist.distances_par_service(base, equipements, services)

manquantes = distances.isna().sum()
print("Distances manquantes par service :", manquantes[manquantes > 0].to_dict() or "aucune")
assert (distances[presence].fillna(0) == 0).all().all(), "Une commune équipée a une distance non nulle."

display(dsc.resume_par_service(distances))

## 6. Table finale

**Objectif.** Rassembler en **une table, une ligne par commune**, tout ce qui sert à l'analyse : identité, surface, population, densité et les 19 distances. C'est le tableau de référence pour la suite du projet.

**Ce que fait le code.**
1. Joint les distances à la base sur le code commune (`base.set_index("code").join(distances)`).
2. Enregistre la table et le tableau de présence au format parquet dans `data/processed/`.

**Sorties.** `data/processed/communes_distances.parquet` et `data/processed/communes_presence.parquet`.

**À vérifier.** Le nombre de lignes de `table` est égal à celui de `base`. Le revenu (Filosofi) sera ajouté à l'étape suivante, avec les corrélations.

In [ ]:
table = base.set_index("code").join(distances)
assert len(table) == len(base)

table.reset_index().to_parquet(PROCESSED / "communes_distances.parquet", index=False)
presence.reset_index().to_parquet(PROCESSED / "communes_presence.parquet", index=False)

print(f"Table finale : {table.shape[0]:,} lignes, {table.shape[1]} colonnes")
display(table.head())

## 7. L'indice d'isolement

### 7.1 Principe

On dispose de 19 distances par commune, qu'on veut **résumer en un seul chiffre** : l'indice d'isolement. On ne peut pas additionner directement les kilomètres : un service rare (maternité) a de grandes distances partout et écraserait un service fréquent (infirmier) dont les distances sont toutes petites.

**Les trois étapes.**
1. **Logarithme** : `d' = ln(1 + d)`, avec `d` en km. Il adoucit les distances extrêmes (100 km compte environ deux fois 10 km, pas dix fois). Le « +1 » permet de traiter les distances nulles (service présent).
2. **Standardisation** (z-score), **service par service** : `z = (d' − moyenne du service) / écart-type du service`, calculés sur l'ensemble des communes. Chaque service est ramené à la même échelle. Un `z` de +1 signifie « un écart-type plus loin que la moyenne des communes pour ce service ».
3. **Somme pondérée** : `I = Σ(w × z) / Σ(w)`. Plus l'indice est élevé, plus la commune est isolée. Diviser par la somme des poids garde la même échelle quel que soit le jeu de poids.

*Remarque.* Chaque commune compte pour **une** unité dans le calcul de la moyenne et de l'écart-type, quelle que soit sa population. Les analyses de population (étape 8) corrigent cet effet.

### 7.2 Les jeux de poids

| Jeu | Définition | Question à laquelle il répond |
|---|---|---|
| `egaux` | tous les services pèsent 1 | indice de référence |
| `vitaux` | urgences, maternité, médecin, pharmacie, santé de proximité comptent double | et si les soins de premier recours et urgents pèsent plus ? |
| `themes` | chaque thème (commerces, santé, enseignement, services publics, transport) pèse autant | et si la santé (8 services) ne dominait pas ? |
| `acp` | première composante principale des 19 `z` | que dit la structure des données elle-même ? |

L'**ACP** (analyse en composantes principales) trouve la combinaison des 19 `z` qui varie le plus d'une commune à l'autre. Son signe est arbitraire : on l'oriente pour qu'un score élevé veuille dire « plus isolé ».

**Ce que fait le code.** Calcule les `z`, construit les 4 indices, affiche les poids et la part de variance expliquée par la première composante.

In [ ]:
z = idx.standardiser(idx.log_distance(distances))

poids = {
    "egaux": idx.poids_egaux(services),
    "vitaux": idx.poids_vitaux(services),
    "themes": idx.poids_themes_equilibres(services),
}
indices = pd.DataFrame({nom: idx.indice_pondere(z, p) for nom, p in poids.items()})
indices["acp"], part_pc1 = idx.indice_acp(z, indices["egaux"])

print(f"Part de la variance expliquée par la 1re composante (ACP) : {part_pc1:.1%}")
display(pd.DataFrame(poids).round(2).T)
display(indices.describe().round(3))

### 7.3 Robustesse : trois variantes

**Objectif.** Montrer que le classement des communes **ne dépend pas** de nos choix discutables. On recalcule l'indice (à poids égaux) en changeant une chose à la fois :

| Variante | Ce qui change | Choix testé |
|---|---|---|
| `sans_pers_agees` | on retire le service « personnes âgées » | le mélange hébergement et services à domicile |
| `positions_AB` | seules les positions de qualité A et B servent aux distances | la fiabilité des coordonnées (la présence reste inchangée) |
| `sans_regle_presence` | la distance est toujours mesurée depuis le centre de la commune, même si le service est dans la commune | la règle « présent = 0 » |

**Comment on compare.** Par la **corrélation de Spearman** entre les indices : elle compare les **classements** des communes. Elle vaut 1 si deux indices classent les communes dans le même ordre. En règle générale, une valeur supérieure à 0,9 indique des classements très proches.

**Ce que fait le code.** Calcule les trois variantes, ajoute leurs colonnes à `indices`, puis affiche la **matrice de Spearman** sous forme de tableau coloré et l'enregistre en figure.

**À vérifier.** Les valeurs de la matrice. Si une variante s'écarte nettement (par exemple 0,7), **il faut le dire dans le rapport** et en discuter la raison : c'est un résultat, pas un échec.

In [ ]:
# Variante 1 : sans « personnes âgées »
services_sans_pa = [s for s in services if s != "personnes_agees"]
indices["sans_pers_agees"] = idx.indice_pondere(z[services_sans_pa], idx.poids_egaux(services_sans_pa))

# Variante 2 : positions de qualité A et B seulement (la présence n'est pas filtrée)
distances_ab, _ = dist.distances_par_service(base, equipements, services, qualites=QUALITES_FIABLES)
print("Distances manquantes avec A/B seulement :", int(distances_ab.isna().sum().sum()))
indices["positions_AB"] = idx.calculer_indice(distances_ab, idx.poids_egaux(services))

# Variante 3 : sans la règle « présent = 0 »
distances_sans, _ = dist.distances_par_service(base, equipements, services, regle_presence=False)
indices["sans_regle_presence"] = idx.calculer_indice(distances_sans, idx.poids_egaux(services))

rho = idx.comparer_classements(indices)
display(rho.round(3))

fig, ax = plt.subplots(figsize=(8, 6.5))
image = ax.imshow(rho, vmin=min(0.5, float(rho.values.min())), vmax=1, cmap="Blues")
ax.set_xticks(range(len(rho)))
ax.set_xticklabels(rho.columns, rotation=45, ha="right")
ax.set_yticks(range(len(rho)))
ax.set_yticklabels(rho.index)
for i in range(len(rho)):
    for j in range(len(rho)):
        ax.text(j, i, f"{rho.iloc[i, j]:.2f}", ha="center", va="center", color="black")
ax.set_title("Corrélation de Spearman entre les indices")
fig.colorbar(image, shrink=0.8)
fig.savefig(FIGURES / "robustesse_spearman.png", dpi=150, bbox_inches="tight")
plt.show()

indices.reset_index().to_parquet(PROCESSED / "indices.parquet", index=False)

## 8. Statistiques descriptives

### 8.1 Les distances, service par service

**Objectif.** Décrire la distribution des distances : combien de communes sont équipées, quelle est la distance médiane, que se passe-t-il dans les cas extrêmes ?

**Ce que fait le code.** Affiche, pour chaque service, la **part de communes équipées**, la **médiane**, la **moyenne**, le 90ᵉ et le 99ᵉ **centile** (la distance au-delà de laquelle se trouvent les 10 % et 1 % de communes les plus éloignées) et le **maximum**. On compare médiane et moyenne : si la moyenne est bien plus grande, quelques communes très éloignées tirent vers le haut (hypothèse H1 : distribution très asymétrique).

Puis un **diagramme en boîtes** par service, sur l'échelle `ln(1 + d)`, triés par médiane. La boîte contient la moitié centrale des communes, la ligne est la médiane, les points sont les cas extrêmes.

**À vérifier.** Les services rares (urgences, maternité, gare, lycée) doivent apparaître en haut du classement des distances ; les services répandus (boulangerie, école) près de 0.

In [ ]:
resume = dsc.resume_par_service(distances)
display(resume.sort_values("mediane_km", ascending=False))

ordre = distances.median().sort_values().index
fig, ax = plt.subplots(figsize=(8, 7))
np.log1p(distances[ordre]).plot.box(vert=False, ax=ax)
ax.set_xlabel("ln(1 + distance en km)")
ax.set_title("Distance au service le plus proche, par service")
fig.savefig(FIGURES / "boites_distances.png", dpi=150, bbox_inches="tight")
plt.show()

### 8.2 La population éloignée

**Objectif.** Passer des **communes** aux **habitants** : quelle part de la population vit dans une commune à plus de 5, 10 ou 20 km d'un service ? Compter les communes surévalue le problème (beaucoup de petites communes, peu d'habitants) ; compter les habitants le chiffre correctement.

**Ce que fait le code.** Pour chaque service et chaque seuil, `dsc.part_population_eloignee` additionne la **population** des communes dont la distance dépasse le seuil, et divise par la population totale. Une commune équipée a une distance de 0 : elle n'est jamais comptée comme éloignée.

**À vérifier.** Les pourcentages doivent décroître quand le seuil augmente (> 5 km ≥ > 10 km ≥ > 20 km). Les valeurs doivent être faibles pour les services courants et plus élevées pour les services rares.

In [ ]:
pop = base.set_index("code")["population"]
eloignee = dsc.part_population_eloignee(distances, pop, SEUILS_KM)
display(eloignee.sort_values(eloignee.columns[0], ascending=False))

### 8.3 L'indice : distribution et communes les plus isolées

**Objectif.** Lire la distribution de l'indice principal (poids égaux) et identifier les communes les plus isolées.

**Ce que fait le code.**
1. Affiche les **quantiles** de l'indice (`describe`), puis son **histogramme**. On s'attend à une distribution asymétrique : beaucoup de communes autour de la moyenne et une longue queue vers les valeurs élevées.
2. Liste les **20 communes ou arrondissements** à l'indice le plus élevé, avec leur population et leur surface.

**À vérifier.** Les communes en tête doivent être plausibles : petites, souvent en montagne ou en zone très peu peuplée. Si la liste est dominée par des cas étonnants (par exemple de très grandes communes bien desservies), c'est probablement un effet du **centre** de la commune ou des contours, à signaler dans les limites.

In [ ]:
indice = indices[INDICE_PRINCIPAL]
display(dsc.decrire_indice(indice).to_frame(f"indice ({INDICE_PRINCIPAL})"))

fig, ax = plt.subplots(figsize=(8, 4.5))
indice.plot.hist(bins=80, ax=ax, color="#1B6F63")
ax.set_xlabel("indice d'isolement (plus élevé = plus isolé)")
ax.set_ylabel("nombre de communes")
ax.set_title("Distribution de l'indice d'isolement")
fig.savefig(FIGURES / "histogramme_indice.png", dpi=150, bbox_inches="tight")
plt.show()

display(dsc.communes_les_plus_isolees(base, indice, n=20))

## 9. Cartes

### 9.1 La carte de l'indice

**Objectif.** Voir **où** se concentre l'isolement (hypothèse H3).

**Ce que fait le code.**
1. Joint l'indice aux polygones `geo` (sur le code commune) pour obtenir une table cartographique.
2. Trace une **carte choroplèthe** : chaque commune est coloriée selon son indice, du clair (proche des services) au foncé (isolé).
3. **Borne l'échelle de couleurs** entre les centiles 2 et 98 : sans cela, quelques valeurs extrêmes écraseraient les nuances pour 95 % des communes.
4. Enregistre l'image dans `figures/carte_indice.png`.

**À vérifier.** Les grandes agglomérations apparaissent claires. Les zones peu denses (montagnes, certaines zones rurales) apparaissent plus foncées, ce qui doit se lire clairement sans être grisé par quelques valeurs extrêmes. Le tracé de ~35 000 polygones prend quelques dizaines de secondes.

In [ ]:
carte = geo.merge(indices.reset_index(), on="code")

vmin, vmax = carte[INDICE_PRINCIPAL].quantile([0.02, 0.98])
fig, ax = plt.subplots(figsize=(10, 10))
carte.plot(column=INDICE_PRINCIPAL, cmap="YlOrRd", vmin=vmin, vmax=vmax, linewidth=0, ax=ax,
           legend=True, legend_kwds={"shrink": 0.6, "label": "indice d'isolement"})
ax.set_axis_off()
ax.set_title("Isolement des communes de France métropolitaine")
fig.savefig(FIGURES / "carte_indice.png", dpi=200, bbox_inches="tight")
plt.show()

### 9.2 Les cartes par service

**Objectif.** Voir **quel service manque, et où** : l'indice résume 19 services, mais deux communes de même indice peuvent manquer de services très différents.

**Ce que fait le code.** Pour les six services de `SERVICES_CARTES`, trace une petite carte de `ln(1 + distance)`. Chaque carte a **sa propre échelle de couleurs** (bornée au 98ᵉ centile), puisque les distances d'une gare et d'une boulangerie n'ont pas la même ampleur. L'image est enregistrée dans `figures/cartes_services.png`.

**À vérifier.** La carte des urgences ou des gares montre de grandes zones foncées, celle de la boulangerie presque aucune. Cela doit rendre visibles les écarts entre services évoqués plus haut.

In [ ]:
carte_services = geo.merge(distances.reset_index(), on="code")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, service in zip(axes.ravel(), SERVICES_CARTES):
    valeurs = np.log1p(carte_services[service])
    carte_services.assign(valeur=valeurs).plot(
        column="valeur", cmap="YlOrRd", vmin=0, vmax=float(valeurs.quantile(0.98)),
        linewidth=0, ax=ax, legend=False)
    ax.set_axis_off()
    ax.set_title(service)
fig.suptitle("Distance au service le plus proche, ln(1 + km)", y=0.94)
fig.savefig(FIGURES / "cartes_services.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Isolement et densité (hypothèse H2)

**Objectif.** Tester l'hypothèse : **plus une commune est peu dense, plus elle est isolée.** Cela paraît évident, mais le chiffrer permet de dire *à quel point*, et de repérer les exceptions.

**Pourquoi la corrélation de Spearman et pas celle de Pearson.** Pearson mesure une relation en ligne droite entre les valeurs. La densité est très asymétrique (de 1 hab./km² à plus de 20 000), donc quelques communes écraseraient tout. Spearman **remplace chaque valeur par son rang** (la 1ʳᵉ, la 2ᵉ, la 3ᵉ…) puis calcule la corrélation des rangs : elle ne dépend ni de l'échelle ni des valeurs extrêmes, et détecte toute relation qui va dans le même sens, droite ou courbe. Elle vaut −1 (sens opposés), 0 (aucun lien) ou +1 (même ordre).

**Entrées.** `indice` (poids égaux, une valeur par commune) et la colonne `densite` de `base` (habitants par km²).

**Ce que fait le code.**
1. Charge le module `analyse` et aligne `indice` et `dens` sur les mêmes codes commune.
2. Calcule le Spearman entre les deux, sur les communes qui ont une densité. La p-value mesure la probabilité d'observer un lien aussi fort par hasard : avec 34 000 communes, elle sera minuscule pour presque tout lien, donc **la valeur de ρ compte plus que la p-value**.
3. Range les communes en **5 classes de densité de même effectif** (quintiles : 20 % des communes les moins denses, etc.) et compare l'indice entre classes avec le **test de Kruskal-Wallis**. Ce test regarde si l'indice est distribué de la même manière dans tous les groupes. Sa **taille d'effet ε²** (de 0 à 1) dit si l'écart est grand : au-dessus de 0,14 on parle d'effet fort.
4. Trace deux graphiques : un nuage de points (log de la densité en abscisse, indice en ordonnée, les points superposés sont comptés dans des cases hexagonales) et des boîtes à moustaches par classe de densité.

**À vérifier.** ρ doit être **négatif** (plus dense, moins isolé). Les médianes de l'indice doivent diminuer d'une classe à l'autre. Si ρ est proche de 0, il faut chercher une erreur (par exemple une densité mal calculée).

In [ ]:
import analyse as an

dens = base.set_index("code")["densite"]
indice_a = indice.reindex(dens.index)

res_dens = an.correlation_spearman(indice_a, dens)
print(f"Spearman indice / densité : rho = {res_dens['rho']:.3f}  (n = {res_dens['n']:,}, p = {res_dens['p_value']:.2g})")

NOMS_DENSITE = ["1 (très peu dense)", "2", "3", "4", "5 (très dense)"]
classe_densite = pd.qcut(dens.rank(method="first"), 5, labels=NOMS_DENSITE)
res_kw, tab_dens = an.comparer_groupes(indice_a, classe_densite)
print(f"Kruskal-Wallis : H = {res_kw['H']:.0f}, p = {res_kw['p_value']:.2g}, taille d'effet epsilon2 = {res_kw['epsilon2']:.3f}")
display(tab_dens)

ok = dens.notna() & (dens > 0)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
hb = axes[0].hexbin(np.log10(dens[ok]), indice_a[ok], gridsize=60, bins="log", cmap="viridis", mincnt=1)
axes[0].set_xlabel("log10 de la densité (habitants par km²)")
axes[0].set_ylabel("indice d'isolement")
axes[0].set_title("Une case = plusieurs communes (échelle logarithmique)")
fig.colorbar(hb, ax=axes[0], label="nombre de communes")
groupes_d = [indice_a[classe_densite == c].dropna() for c in NOMS_DENSITE]
axes[1].boxplot(groupes_d, showfliers=False)
axes[1].set_xticks(range(1, len(NOMS_DENSITE) + 1))
axes[1].set_xticklabels(NOMS_DENSITE)
axes[1].set_xlabel("classe de densité")
axes[1].set_title("Indice par classe de densité")
axes[1].tick_params(axis="x", rotation=20)
fig.savefig(FIGURES / "densite_indice.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Isolement et revenu (hypothèse H4)

### 11.1 Ajouter le revenu médian

**Objectif.** Disposer du **niveau de vie médian** de chaque commune (indicateur `MED_SL` de Filosofi 2023).

**Particularité du fichier.** Il est en **format long** : une ligne par couple (territoire, indicateur), avec tous les niveaux géographiques mélangés. Il contient plusieurs centaines de milliers de lignes, dont la plupart n'ont pas de valeur : ce sont des indicateurs non publiés à l'échelle communale. Il faut donc filtrer.

**Ce que fait le code.**
1. `an.lire_revenu` ne garde que les lignes dont le type de territoire est une commune (`COM`) ou un arrondissement (`ARM`), et l'indicateur `MED_SL`. Les valeurs absentes (secret statistique `s`, donnée manquante `vm`) deviennent des cases vides, et les codes sont remis sur 5 caractères.
2. `an.fusionner_revenu` joint le revenu à la base par **jointure à gauche** : aucune commune de la base ne disparaît, celles sans revenu restent avec une case vide. Un **journal** compte les communes sans revenu et les codes du fichier absents de la base.

**À vérifier.** Environ 11 % de communes sans revenu (petites communes, secret statistique). Quelques codes du fichier hors de la base sont possibles (fichier en géographie 2026, base en géographie 2025). Si le journal montre 0 revenu trouvé, le type de territoire ne s'appelle pas `COM` dans votre fichier : affichez `pd.read_csv(..., sep=";", dtype=str)["GEO_OBJECT"].value_counts()`.

In [ ]:
FICHIER_REVENU = RAW / "Revenus.csv"

revenu = an.lire_revenu(FICHIER_REVENU)
base_rev, journal_rev = an.fusionner_revenu(base, revenu)
print("Journal de la fusion du revenu :", journal_rev)
print(f"Part des communes sans revenu : {journal_rev['communes_sans_revenu'] / journal_rev['communes_base']:.1%}")

revenu_s = base_rev.set_index("code")["revenu_median"]
display(revenu_s.describe().round(0).to_frame("niveau de vie médian (€)"))

### 11.2 Les communes sans revenu sont-elles différentes ?

**Objectif.** Vérifier si les valeurs manquantes créent un **biais**. Le secret statistique frappe les petites communes, souvent isolées : si on les écarte, on risque de sous-estimer le lien entre isolement et revenu.

**Ce que fait le code.** Sépare les communes en deux groupes (avec revenu, sans revenu), compare l'indice d'isolement et la population de chacun avec le test de Kruskal-Wallis (ici entre deux groupes), puis donne la part de la **population** concernée.

**À vérifier.** Si les communes sans revenu sont nettement plus isolées (indice médian plus élevé, effet non négligeable), il faut l'écrire dans les limites du rapport. Leur part de population doit rester petite.

In [ ]:
a_revenu = pd.Series(np.where(revenu_s.notna(), "avec revenu", "sans revenu"), index=revenu_s.index)

res_man, tab_man = an.comparer_groupes(indice.reindex(a_revenu.index), a_revenu)
print(f"Kruskal-Wallis (indice, avec/sans revenu) : p = {res_man['p_value']:.2g}, epsilon2 = {res_man['epsilon2']:.3f}")
display(tab_man)

pop_s = base.set_index("code")["population"]
print(f"Part de la population dans les communes sans revenu : "
      f"{pop_s[a_revenu == 'sans revenu'].sum() / pop_s.sum():.2%}")

### 11.3 Le lien entre isolement et revenu

**Objectif.** Mesurer si les communes plus isolées sont plus pauvres (ou plus riches), et si l'effet est grand.

**Ce que fait le code.**
1. **Spearman** entre l'indice et le revenu médian, sur les communes qui ont un revenu.
2. **Quintiles de revenu** : 5 groupes de même effectif, du revenu le plus bas au plus haut, puis **Kruskal-Wallis** et sa taille d'effet ε² (voir le §10), avec l'indice médian de chaque groupe.
3. Un **diagramme en boîtes** de l'indice par quintile de revenu.

**Comment lire le résultat.** Un ρ négatif voudrait dire « plus isolé, plus pauvre » ; positif, « plus isolé, plus riche ». Une corrélation **n'est pas une causalité** : elle dit que les deux vont ensemble, pas que l'un provoque l'autre.

**À vérifier.** Le signe, l'ordre de grandeur de ρ (probablement faible à modéré), et que les médianes de l'indice évoluent de façon cohérente avec le quintile.

In [ ]:
indice_r = indice.reindex(revenu_s.index)

res_rev = an.correlation_spearman(indice_r, revenu_s)
print(f"Spearman indice / revenu : rho = {res_rev['rho']:.3f}  (n = {res_rev['n']:,}, p = {res_rev['p_value']:.2g})")

NOMS_REVENU = ["1 (plus bas)", "2", "3", "4", "5 (plus haut)"]
quintile_revenu = pd.qcut(revenu_s.rank(method="first"), 5, labels=NOMS_REVENU)
res_kw_rev, tab_rev = an.comparer_groupes(indice_r, quintile_revenu)
print(f"Kruskal-Wallis : H = {res_kw_rev['H']:.0f}, p = {res_kw_rev['p_value']:.2g}, epsilon2 = {res_kw_rev['epsilon2']:.3f}")
display(tab_rev)

fig, ax = plt.subplots(figsize=(8, 5))
groupes = [indice_r[quintile_revenu == q].dropna() for q in NOMS_REVENU]
ax.boxplot(groupes, showfliers=False)
ax.set_xticks(range(1, len(NOMS_REVENU) + 1))
ax.set_xticklabels(NOMS_REVENU)
ax.set_xlabel("quintile de niveau de vie médian")
ax.set_ylabel("indice d'isolement")
ax.set_title("Indice d'isolement selon le revenu médian de la commune")
fig.savefig(FIGURES / "revenu_indice.png", dpi=150, bbox_inches="tight")
plt.show()

### 11.4 Le piège de la densité

**Objectif.** Vérifier que le lien observé entre isolement et revenu **n'est pas simplement un effet de la densité**. La densité est liée à l'isolement (§10) *et* au revenu (les villes ont des revenus médians différents des campagnes). Une variable qui influence les deux à la fois peut créer un lien apparent entre elles : c'est un **facteur de confusion**.

**Ce que fait le code.**
1. Calcule la corrélation de Spearman entre le revenu et la densité.
2. Refait la corrélation indice / revenu **séparément dans chacune des 5 classes de densité**. Dans chaque classe, la densité varie peu : si le lien persiste, il ne vient pas que de la densité.

**Comment lire.** Si ρ s'effondre vers 0 dans les classes, le lien global était surtout dû à la densité. S'il garde le même signe et une taille comparable, l'isolement a un lien propre avec le revenu.

In [ ]:
res_rev_dens = an.correlation_spearman(revenu_s, dens.reindex(revenu_s.index))
print(f"Spearman revenu / densité : rho = {res_rev_dens['rho']:.3f}  (n = {res_rev_dens['n']:,})")

classe_d = classe_densite.reindex(revenu_s.index)
lignes = {c: an.correlation_spearman(indice_r[classe_d == c], revenu_s[classe_d == c]) for c in NOMS_DENSITE}
print("\nCorrélation indice / revenu à l'intérieur de chaque classe de densité :")
display(pd.DataFrame(lignes).T.round(3))

## 12. Typologie : quels types d'isolement ?

**Objectif.** L'indice dit *à quel point* une commune est isolée, pas *de quoi*. Deux communes de même indice peuvent manquer de services très différents. On cherche donc des **profils types** : des groupes de communes qui se ressemblent par la manière dont elles sont éloignées des services.

**Méthode.** On part des 19 écarts standardisés `z` de chaque commune (§7). Le **k-means** forme `k` groupes en répétant deux étapes : rattacher chaque commune au centre de groupe le plus proche, puis recalculer chaque centre comme la moyenne de son groupe. Il faut fixer `k` à l'avance. Pour le choisir, on regarde deux indicateurs :
- l'**inertie** (somme des carrés des écarts aux centres) : elle baisse toujours quand `k` augmente ; on cherche le « coude » où elle cesse de baisser vite ;
- la **silhouette** (de −1 à 1) : elle dit si les groupes sont bien séparés ; plus elle est haute, mieux c'est.

On choisit `k` par jugement, entre 4 et 6 pour rester lisible, en s'aidant de ces deux courbes.

**Ce que fait le code.**
1. `an.choisir_k` calcule inertie et silhouette (sur un échantillon de 5 000 communes, pour aller vite) pour k de 2 à 8, et on les trace.
2. On fixe `N_GROUPES`, puis `an.typologie` lance le k-means et **numérote les groupes du mieux desservi (0) au moins bien desservi** (la numérotation brute du k-means est aléatoire). Elle calcule aussi le plan des deux premières composantes principales (PC1, PC2) pour dessiner les groupes.
3. On calcule le **profil moyen de chaque groupe** (la moyenne des `z` de chaque service) et on l'affiche en carte de chaleur : rouge = plus éloigné que la moyenne, bleu = plus proche.
4. Un tableau résume chaque groupe : nombre de communes, part de la population, indice, densité et revenu médians.
5. Une **carte** colorie chaque commune selon son groupe.

**À vérifier et à faire à la main.** Lisez la carte de chaleur et **donnez un nom** à chaque groupe (par exemple « bien desservies », « manque de santé », « isolement général fort »). Renseignez-les dans `NOMS_GROUPES`.

In [ ]:
cout_k = an.choisir_k(z)
display(cout_k)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cout_k["inertie"].plot(marker="o", ax=axes[0], title="Inertie (chercher le coude)")
cout_k["silhouette"].plot(marker="o", ax=axes[1], title="Silhouette (plus haut = mieux séparé)")
fig.savefig(FIGURES / "choix_k.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
N_GROUPES = 5        # à ajuster d'après les courbes ci-dessus
res_typo, profils, var_pca = an.typologie(z, N_GROUPES)
print(f"Variance expliquée par PC1 et PC2 : {var_pca.round(3)}")

# Profils : moyenne des z de chaque service dans chaque groupe
fig, ax = plt.subplots(figsize=(12, 3.5 + 0.4 * N_GROUPES))
im = ax.imshow(profils, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(profils.shape[1]))
ax.set_xticklabels(profils.columns, rotation=60, ha="right")
ax.set_yticks(range(N_GROUPES))
ax.set_yticklabels([f"groupe {g}" for g in profils.index])
for i in range(profils.shape[0]):
    for j in range(profils.shape[1]):
        ax.text(j, i, f"{profils.iloc[i, j]:.1f}", ha="center", va="center", fontsize=8)
ax.set_title("Profil de chaque groupe (rouge = plus éloigné que la moyenne)")
fig.colorbar(im, shrink=0.8)
fig.savefig(FIGURES / "profils_groupes.png", dpi=150, bbox_inches="tight")
plt.show()

# Résumé par groupe
g = res_typo["groupe"]
resume_groupes = pd.DataFrame({
    "communes": g.value_counts().sort_index(),
    "part_population_%": (pop_s.groupby(g).sum() / pop_s.sum() * 100).round(1),
    "indice_median": indice.groupby(g).median().round(2),
    "densite_mediane": dens.groupby(g).median().round(1),
    "revenu_median": revenu_s.groupby(g).median().round(0),
})
display(resume_groupes)

In [ ]:
# À COMPLÉTER après lecture des profils : un nom par groupe
NOMS_GROUPES = {i: f"Groupe {i}" for i in range(N_GROUPES)}
# Exemple : NOMS_GROUPES = {0: "Bien desservies", 1: "...", 2: "...", 3: "...", 4: "Isolement général fort"}

carte_typo = geo.merge(res_typo.rename_axis("code").reset_index(), on="code")
carte_typo["type"] = carte_typo["groupe"].map(NOMS_GROUPES)

fig, ax = plt.subplots(figsize=(10, 10))
carte_typo.plot(column="type", categorical=True, cmap="YlOrRd", linewidth=0, ax=ax, legend=True,
                legend_kwds={"loc": "lower left", "title": "Type de commune"})
ax.set_axis_off()
ax.set_title("Typologie des communes selon leur profil d'éloignement")
fig.savefig(FIGURES / "carte_typologie.png", dpi=200, bbox_inches="tight")
plt.show()

# Plan PC1 x PC2 (échantillon de 8 000 communes pour la lisibilité)
echantillon = res_typo.sample(min(8000, len(res_typo)), random_state=0)
fig, ax = plt.subplots(figsize=(8, 6))
nuage = ax.scatter(echantillon["pc1"], echantillon["pc2"], c=echantillon["groupe"], cmap="YlOrRd", s=6, alpha=0.6)
ax.set_xlabel(f"PC1 ({var_pca[0]:.0%} de la variance)")
ax.set_ylabel(f"PC2 ({var_pca[1]:.0%} de la variance)")
ax.set_title("Les groupes dans le plan des deux premières composantes")
fig.colorbar(nuage, label="groupe")
fig.savefig(FIGURES / "plan_acp.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Autocorrélation spatiale (bonus, hypothèse H3)

**Objectif.** Tester que l'isolement est **groupé géographiquement** : les communes isolées sont-elles voisines les unes des autres, plutôt que dispersées au hasard ? La carte du §9.1 le suggère (Alpes, Pyrénées, nord-est) ; ici on le **mesure**.

**Le I de Moran global.** Pour chaque commune, on compare sa valeur à la moyenne de ses voisines. Le I vaut environ **0** si la répartition est aléatoire, tend vers **+1** si les valeurs semblables se regroupent, et vers **−1** si les valeurs opposées alternent. La significativité vient de **permutations** : on mélange la carte au hasard 199 fois, on calcule le I à chaque fois, et on regarde si notre valeur sort de la distribution obtenue.

**Qui est voisin ?** Les **8 communes les plus proches** (distance entre centres). Ce choix évite le problème des communes de tailles très différentes, avec très peu de voisines pour certaines et beaucoup pour d'autres.

**Les cartes LISA (I de Moran local).** On classe chaque commune en :
- **HH** : isolée, entourée de communes isolées (un « pôle » d'isolement) ;
- **LL** : bien desservie, entourée de communes bien desservies ;
- **HL / LH** : une commune isolée au milieu de communes bien desservies, ou l'inverse ;
- **ns** : pas de structure locale significative.

**Ce que fait le code.**
1. Récupère les coordonnées des centres, avec les mêmes codes que l'indice.
2. Calcule le I de Moran global de l'indice, puis un **contrôle** sur l'indice mélangé au hasard (qui doit donner I ≈ 0, ce qui valide la méthode).
3. Calcule les classes LISA, les compte, et trace la carte.

**À vérifier.** Un I nettement positif (probablement entre 0,5 et 0,8) pour l'indice réel et ≈ 0 pour l'indice mélangé. Les communes HH doivent se trouver dans les zones foncées de la carte du §9.1. Si les bibliothèques `esda` et `libpysal` ne sont pas installées, la cellule l'indique : `pip install esda libpysal`. Le calcul prend d'une à quelques minutes.

In [ ]:
K_VOISINS = 8
lisa = None
try:
    xy = base.set_index("code").loc[indice.index, ["x", "y"]]

    moran = an.moran_global(indice, xy["x"], xy["y"], k=K_VOISINS)
    print(f"I de Moran global (indice) : {moran['I']:.3f}  (espérance sous le hasard {moran['esperance']:.4f}, p = {moran['p_sim']:.3g})")

    melange = pd.Series(indice.sample(frac=1, random_state=0).to_numpy(), index=indice.index)
    controle = an.moran_global(melange, xy["x"], xy["y"], k=K_VOISINS)
    print(f"Contrôle sur l'indice mélangé au hasard : I = {controle['I']:.3f} (doit être proche de 0)")

    lisa = an.moran_local(indice, xy["x"], xy["y"], k=K_VOISINS)
    display(lisa.value_counts().to_frame("communes"))

    from matplotlib.patches import Patch
    couleurs = {"HH": "#B2182B", "LL": "#2166AC", "HL": "#EF8A62", "LH": "#67A9CF", "ns": "#E5E5E5"}
    carte_l = geo.merge(lisa.rename_axis("code").reset_index(), on="code")
    fig, ax = plt.subplots(figsize=(10, 10))
    for classe, couleur in couleurs.items():
        carte_l[carte_l["lisa"] == classe].plot(color=couleur, linewidth=0, ax=ax)
    ax.legend(handles=[Patch(color=c, label=k) for k, c in couleurs.items()], loc="lower left")
    ax.set_axis_off()
    ax.set_title("Regroupements spatiaux de l'isolement (LISA)")
    fig.savefig(FIGURES / "carte_lisa.png", dpi=200, bbox_inches="tight")
    plt.show()
except ImportError as erreur:
    print("Section ignorée, bibliothèque manquante :", erreur, "-> pip install esda libpysal")

## 14. Conclusion

### 14.1 Le tableau final

**Objectif.** Rassembler en un seul fichier tout ce qui sert à la suite : indice, rang en centile, groupe de la typologie, classe LISA, population, densité, revenu.

**Ce que fait le code.** Construit le tableau (une ligne par commune), ajoute le **centile** de l'indice (« plus isolée que X % des communes »), qui est plus parlant qu'un z-score négatif ou positif, puis l'enregistre dans `data/processed/communes_analyse.parquet`.

In [ ]:
final = base.set_index("code")[["nom", "population", "surface_km2", "densite"]].copy()
final["revenu_median"] = revenu_s
final["indice"] = indice
final["indice_centile"] = (indice.rank(pct=True) * 100).round(1)
final["groupe"] = res_typo["groupe"]
if lisa is not None:
    final["lisa"] = lisa

final.reset_index().to_parquet(PROCESSED / "communes_analyse.parquet", index=False)
print(f"Enregistré : {final.shape[0]:,} communes, {final.shape[1]} colonnes")
display(final.sort_values("indice", ascending=False).head(10))

### 14.2 Résumé des hypothèses

À remplir avec **vos résultats** après exécution (chaque ligne renvoie à la cellule qui donne le chiffre).

| Hypothèse | Chiffre à reporter | Où le lire | Verdict |
|---|---|---|---|
| **H1** : la distribution des distances est très asymétrique (quelques communes très éloignées) | médiane et moyenne, centile 99 | §8.1 | |
| **H2** : plus la commune est peu dense, plus elle est isolée | ρ de Spearman indice / densité ; ε² du Kruskal-Wallis | §10 | |
| **H3** : l'isolement est groupé géographiquement | I de Moran, nombre de communes HH | §13 | |
| **H4** : lien entre isolement et revenu | ρ indice / revenu ; ρ à densité constante | §11.3 et §11.4 | |

### 14.3 Les limites à écrire dans le rapport

1. **Distance à vol d'oiseau.** Ce n'est pas un temps de trajet : une commune de montagne à 8 km à vol d'oiseau peut être à 40 minutes de route.
2. **Point de départ = centre du polygone.** Dans une grande commune, l'habitant peut vivre loin de ce centre.
3. **Règle « présent = 0 ».** Une grande commune est comptée comme équipée même si l'équipement est loin de certains habitants. La variante `sans_regle_presence` mesure l'effet de ce choix (§7.3).
4. **Coordonnées manquantes** (environ 4 % des équipements) : utilisées pour la présence mais pas pour les distances.
5. **Le panier de 19 services est un choix.** Les regroupements, notamment « personnes âgées », sont discutables ; les variantes du §7.3 montrent que le classement résiste à ces choix.
6. **Métropole seulement** : la Corse est incluse, les départements d'outre-mer sont exclus.
7. **Géographies différentes** : contours et BPE sont en géographie 2025, la population municipale aussi, mais le revenu Filosofi est en géographie 2026.
8. **Revenu manquant pour environ 11 % des communes** (secret statistique), des petites communes plutôt isolées (§11.2).
9. **Corrélation n'est pas causalité**, et un résultat sur des communes ne dit rien sur le comportement des individus qui y vivent (c'est la limite dite « écologique »).
10. **Chaque commune compte pour une unité** dans la standardisation, quelle que soit sa population : l'indice décrit des territoires, pas des habitants. Le §8.2 donne la lecture par population.

### 14.4 Fichiers produits

| Fichier | Contenu |
|---|---|
| `data/interim/equipements.parquet` | équipements du panier, métropole |
| `data/processed/communes_distances.parquet` | une ligne par commune : population, densité, 19 distances |
| `data/processed/communes_presence.parquet` | présence de chaque service |
| `data/processed/indices.parquet` | les indices et leurs variantes |
| `data/processed/communes_analyse.parquet` | indice, centile, groupe, LISA, densité, revenu |
| `figures/*.png` | graphiques et cartes |